In [1]:
import requests
import json
import pandas as pd
import folium
import time
from datetime import datetime
import os
from pathlib import Path

url_race='https://sarl.ingenium.net.au/racelog?racenr=38602'
html=os.path.join(str(Path.home()), 'Documents', 'boats.html')
mine='Petsamo'

In [2]:
res=requests.get(url_race)

In [3]:
boats = res.json()['result']

In [4]:
boats

{'Encelade-Voyage Voyage': {'heading': 41,
  'lat_dec': -28.6192,
  'lon_dec': 153.7223,
  'lastreport_heading': '41',
  'lastreport_speed': '5.3',
  'sail': '117873',
  'status': None,
  'wind': '356°, 15.9kn.',
  'ubtname': 'Voyage Voyage',
  'usrname': 'Encelade',
  'racetime': -1041372000,
  'teamnr': 0,
  'resultdescr': "S28°37.154' E153°43.338', Hdg: 41°, Spd: 5.3kn., 3049.7nm. to mark 2/13",
  'finished': 'false',
  'teaname': None,
  'usrnr': 21144,
  'rank': '1',
  'btptype': "45' Ketch",
  'racing': 1,
  'timestamp': 1672859485000.0,
  'trackdistance': '368.492',
  'points': '0',
  'nextmarknr': '2',
  'distancetonextmark': '3049.71379719886',
  'track': [[-28.6192, 153.7223],
   [-28.9796, 153.6909],
   [-29.7408, 153.843],
   [-30.1837, 153.6968],
   [-30.3274, 153.5014],
   [-30.7883, 153.5554],
   [-31.1288, 153.4828],
   [-31.388, 153.4828],
   [-31.509, 153.4467],
   [-31.6306, 153.1994],
   [-31.7385, 152.8927],
   [-32.5503, 152.6125],
   [-32.8815, 152.5129],
   [-32

In [5]:
data=[(boats[boat]['ubtname'],
       boats[boat]['lat_dec'],
       boats[boat]['lon_dec'],
       boats[boat]['rank'])
      for boat in list(boats.keys()) if 'lat_dec' in boats[boat]]

In [6]:
df=pd.DataFrame(data, columns=['Boat', 'Lat', 'Lon', 'Rank'])
df['Rank']=df['Rank'].astype('int64')
df.sort_values('Rank', ascending=True, inplace=True)
df.reset_index(inplace=True)

In [7]:
colors=['red', 'green', 'darkblue', 'orange', 'pink', 'darkgreen',
        'beige', 'darkred', 'purple', 'darkpurple']

In [8]:
center=(df.loc[0, 'Lat'], df.loc[0, 'Lon'])
mymap=folium.Map(location=center, zoom_start=7)
for idx in df.index:
    lat=df.loc[idx, 'Lat']
    lon=df.loc[idx, 'Lon']
    name=df.loc[idx, 'Boat'].upper()
    rank=df.loc[idx, 'Rank']
    if idx < len(colors):
        color=colors[idx]
    else:
        color='lightgray'
    if df.loc[idx, 'Boat']==mine:
        color='lightblue'
    popup=str(lat) + ', ' + str(lon)
    folium.Marker([lat, lon], popup=name + ' ' + str(rank), icon=folium.Icon(color=color, icon='sailboat', prefix='fa')).add_to(mymap)

In [10]:
mymap.save(html)